In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from dataclasses import dataclass

@dataclass
class EmailContext:
    email_address: str= "zakariaahmed29@example.com"
    passward: str= "password123"

In [3]:
from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool

In [4]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_inbox() ->str:
    """check the inbox for recent emails"""
    return """
    Hi zakaria, 
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (zakariaahmed29@example.com)
    """


@tool
def send_email(to: str, subject: str, body: str) ->str:
    """send a response email"""
    return f"Email sent to {to} with the subject {subject} and the body {body}"

@tool
def authenticate(email: str, passward: str, runtime: ToolRuntime) ->Command:
    """Authenticate the user with the given email and password"""
    if email== runtime.context.email_address and passward == runtime.context.passward:
        return Command(update={
            "authenticated": True,
            "messages":[ToolMessage("Successfully authenticated",tool_call_id= runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage("Authentication failed", tool_call_id= runtime.tool_call_id)]
        })


In [5]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, handler:callable[[ModelRequest], ModelResponse]) ->ModelResponse:
    """Allow read inbox and send email tools only if user provides correct email and password"""
    authenticated= request.state.get("authenticated")

    if authenticated:
        tools=[send_email, check_inbox]

    else:
        tools= [authenticate]  
    request= request.override(tools= tools)  
    return handler(request)   


In [6]:
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = """You are a helpful assistant that can check the inbox and send emails. 
Your first step after authentication is to check the inbox."""
unauthenticated_prompt = "You are a helpful assistant that can authenticate users."

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) ->str:
    """Generate system prompt based on authentication status"""
    authenticated=request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt


In [7]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    "gpt-5-nano",
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call, 
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
            })
        ]
    )

In [8]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="zakariaahmed29@example.com, password123")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

d:\langchain-course\venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=EmailContext(email_addres... passward='password123'), input_type=EmailContext])
  function=lambda v, h: h(v), schema=original_schema
d:\langchain-course\venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=EmailContext(email_addres... passward='password123'), input_type=EmailContext])
  return self.__pydantic_serializer__.to_python(


Nice catch. I found an email from Jane inviting you to coffee next week.

Would you like me to reply with a draft? Here’s a ready-to-send option you could use (you can customize times or location):

Subject: Re: Coffee next week
Hi Jane,
That sounds great! I’d love to catch up. I’m free Tuesday afternoon or Thursday morning next week. How about Tuesday at 3:00 PM or Thursday at 11:00 AM? If you have a preferred coffee spot, I’m happy to meet there.
Looking forward to it,
Zakaria

Important: I don’t yet have Jane’s email address to send this. Please provide the recipient email (or confirm the thread’s address if I should reply to Jane’s message). Also, tell me your preferred time slots and any location preferences, and I’ll send the final email right away.


In [9]:
response = agent.invoke(
    {"messages": [HumanMessage(content="any draft is fine. don't check back.")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

Here’s a ready-to-send draft you can use. Copy it into your email client and send to Jane’s address (the recipient email you have for Jane).

Subject: Re: Coffee next week
Hi Jane,
That sounds great! I’d love to catch up. I’m available Tuesday afternoon or Thursday morning next week. How about Tuesday at 3:00 PM or Thursday at 11:00 AM? If you have a preferred coffee spot, I’m happy to meet there.
Looking forward to it,
Zakaria

If you want me to send automatically, please provide Jane’s email address (or confirm the thread’s address).


In [14]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="jane@example.com")]},
    context=EmailContext(),
    config=config
)

print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi Jane,
That sounds great! I’d love to catch up. I’m available Tuesday afternoon or Thursday morning next week. How about Tuesday at 3:00 PM or Thursday at 11:00 AM? If you have a preferred coffee spot, I’m happy to meet there.
Looking forward to it,
Zakaria


In [15]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}  
    ), 
    config=config 
)
print(response["messages"][-1].content)

Email sent successfully.

To: jane@example.com
Subject: Re: Coffee next week
Body:
Hi Jane,
That sounds great! I’d love to catch up. I’m available Tuesday afternoon or Thursday morning next week. How about Tuesday at 3:00 PM or Thursday at 11:00 AM? If you have a preferred coffee spot, I’m happy to meet there.
Looking forward to it,
Zakaria

Would you like me to:
- Set a reminder to follow up if Jane doesn’t reply within, say, 3–5 days?
- Draft a polite follow-up email in case there’s no response?
- Add a calendar hold for the proposed times?


In [16]:
from pprint import pprint

pprint(response)

{'authenticated': True,
 'messages': [HumanMessage(content='zakariaahmed29@example.com, password123', additional_kwargs={}, response_metadata={}, id='705a07bb-d0b2-45d8-999f-bf72a6bb1801'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 226, 'prompt_tokens': 155, 'total_tokens': 381, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DrYnJAWY0Fho9XXv6vqf50EQ0RBha', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ed310-0e92-79b0-a477-718cd9676ffa-0', tool_calls=[{'name': 'authenticate', 'args': {'email': 'zakariaahmed29@example.com', 'passward': 'password123'}, 'id': 'call_8v63m7zRyxQBdnjIM3psXbI7', 